# Day 12 — Contextual Precision: Is the Ranking Right?

**Module 3 · RAG Evaluation**

A retriever does not just need to find relevant information.

It should rank the **most useful information first**.

Today we learn:

> **Contextual Precision measures whether relevant context is ranked highly.**

```text
Question
   ↓
Retriever
   ↓
1. Relevant chunk   ← ideal
2. Less relevant
3. Irrelevant

In [1]:
import os

from dotenv import load_dotenv
from deepeval.models import OpenAIModel
from deepeval.test_case import LLMTestCase
from deepeval.metrics import ContextualPrecisionMetric
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig

load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found."

judge = OpenAIModel(
    model="gpt-4.1-mini",
    temperature=0,
)

async_config = AsyncConfig(max_concurrent=2)

print("Judge:", judge.get_model_name())

Judge: gpt-4.1-mini


## 1. Create a RAG Evaluation Case

For retrieval evaluation, the important fields are:

- `input` → the user's question
- `expected_output` → what the answer should contain
- `retrieval_context` → what the retriever actually returned

The final answer is not the focus today.
We are evaluating the **retriever**.

In [2]:
question = "What is a token in an LLM?"

expected_answer = (
    "A token is a basic unit of text processed by an LLM. "
    "Input and output are measured in tokens."
)

tokens_context = (
    "Large language models split text into tokens. "
    "Tokens are the basic units processed by an LLM, "
    "and both input and output are measured in tokens."
)

embeddings_context = (
    "An embedding is a vector representation of text. "
    "Texts with similar meanings have vectors that are close together, "
    "which enables semantic search."
)

rag_context = (
    "RAG grounds an LLM's answer in external documents. "
    "The system retrieves relevant information and provides it to the LLM "
    "as context, which can reduce hallucinations."
)

## 2. Same Context, Different Ranking

We will create two cases.

**Case A:** The relevant document is first.

**Case B:** The relevant document is buried at the bottom.

The actual documents are the same.
Only their order changes.

In [3]:
case_first = LLMTestCase(
    input=question,
    expected_output=expected_answer,
    retrieval_context=[
        tokens_context,
        embeddings_context,
        rag_context,
    ],
)

case_buried = LLMTestCase(
    input=question,
    expected_output=expected_answer,
    retrieval_context=[
        embeddings_context,
        rag_context,
        tokens_context,
    ],
)

print("Case A - relevant context position: 1")
print("Case B - relevant context position: 3")

Case A - relevant context position: 1
Case B - relevant context position: 3


## 3. Run Contextual Precision

Contextual Precision asks whether useful information appears early in the retrieved list.

This matters because real RAG systems usually pass only the **top-k** retrieved chunks to the generator.

If the useful document is ranked too low, it may never reach the LLM.

In [4]:
precision = ContextualPrecisionMetric(
    model=judge,
    threshold=0.5,
)

results = evaluate(
    test_cases=[case_first, case_buried],
    metrics=[precision],
    async_config=async_config,
)

for result in results.test_results:
    metric_result = result.metrics_data[0]

    print(
        f"{result.name}: "
        f"score={metric_result.score:.2f}, "
        f"success={metric_result.success}"
    )
    print(f"reason: {metric_result.reason}")
    print()

✨ You're running DeepEval's latest Contextual Precision Metric! (using gpt-4.1-mini, strict=False, 
async_mode=True)...

c:\Users\T14s\AppData\Local\Programs\Python\Python311\Lib\site-packages\rich\live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:              What is a token in an LLM?                                                           │
│  │     Actual Output:      None                                                                                 │
│  │     Expected Output:    A token is a basic unit of text processed by an LLM. Input and output are            │
│  │                         measured in tokens.                                                                  │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric               ┃ Score ┃ Threshold ┃ Reason                                                │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Contextual Precision │ 0.33  │ 0.50      │ The score is 0.33 because the first two nodes in      │
│              │                      │       │           │ retrieval contexts, ranked 1st and 2nd, are           │
│              │                      │       │           │ irrelevant as they discuss embeddings, semantic       │
│              │                      │       │           │ search, and RAG without defining a token. The         │
│              │                      │       │           │ relevant node is ranked 3rd and clearly defines a     │
│              │                      │       │           │ token in an LLM, so it is correctly identified but    │
│              │                      │       │           │ ranked lower than irrelevant nodes, lowering the      │
│              │                      │       │           │ score.                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                          ┃ Average Score        ┃ Pass Rate                                 ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Contextual Precision            │ 0.67                 │ 50.00% | passed=1 | failed=1              │ 2         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=870623;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 6.14s | token cost: 0.0015804000000000003 USD)
» Test Results (2 total tests):
   » Pass Rate: 50.0% | Passed: 1 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

test_case_1: score=0.33, success=False
reason: The score is 0.33 because the first two nodes in retrieval contexts, ranked 1st and 2nd, are irrelevant as they discuss embeddings, semantic search, and RAG without defining a token. The relevant node is ranked 3rd and clearly defines a token in an LLM, so it is correctly identified but ranked lower than irrelevant nodes, lowering the score.

test_case_0: score=1.00, success=True
reason: The score is 1.00 because the first node in retrieval contexts clearly defines a token in an LLM, stating 'Tokens are the basic units processed by an LLM, and both input and output are measured in tokens,' which is directly relevant. The subsequent nodes, ranked lower, discuss unrelated topics like embeddings and RAG, ensuring that irrelevant nodes are properly ranked below the relevant one. This perfect ordering justifies the top score.



## 4. What Did We Learn?

Contextual Precision is primarily about **ranking quality**.

```text
High precision
→ relevant context appears early

Low precision
→ useful context is buried behind less useful context

# Day 12 — Key Takeaways

- Contextual Precision evaluates **retrieval ranking**.
- Relevant context appearing earlier is better.
- `retrieval_context` contains the documents returned by the retriever.
- `expected_output` helps the metric determine what information is actually needed.
- Poor ranking matters because RAG systems usually use only the top-k results.

**Next:**

> Contextual Precision asks **"Did we rank the useful information highly?"**